# Long-Term Memory with LangMem — Background Processing

## What this notebook teaches

Most LLM chatbots are **stateless** — they forget everything the moment the conversation ends. This notebook shows how to give a chatbot **persistent long-term memory** that survives across sessions.

## How it works

```
User sends message
       │
       ▼
 Chat function (fast)  ──► Returns response immediately
       │
       ▼ (in background, after a delay)
 ReflectionExecutor
       │
       ▼
 Memory Manager (LLM)  ──► Reads conversation, extracts key facts
       │
       ▼
 InMemoryStore         ──► Saves facts as searchable memories
```

**Key idea:** Memory extraction runs *after* the response is sent, so the user never waits for it.

In [1]:
import asyncio

from langchain.chat_models import init_chat_model
from langgraph.func import entrypoint
from langgraph.store.memory import InMemoryStore
from langmem import ReflectionExecutor, create_memory_store_manager

## Step 1: Create the Memory Store

`InMemoryStore` is a vector database that lives in RAM. It stores memories as embeddings so you can do **semantic search** — find memories by meaning, not just exact keywords.

- `dims=768` — the vector size produced by Gemini's embedding model
- `embed` — the model used to convert text memories into vectors

In [2]:
store = InMemoryStore(
    index={
        "dims": 768,                              # Must match the embedding model's output size
        "embed": "google_genai:gemini-embedding-001",  # Model used to embed memories into vectors
    }
)

## Step 2: Set Up the Memory Manager and Executor

**`create_memory_store_manager`** is an LLM-powered "librarian". It reads a conversation and decides which facts are worth remembering — e.g. "User's dog is named Fido".

- `namespace=("memories",)` — organises memories into named folders (like a directory path). You could use `("user_123", "memories")` to keep memories per-user.

**`ReflectionExecutor`** wraps the memory manager to run it asynchronously:
- Your chat function returns a response immediately
- The executor processes the conversation in the background after a delay
- If new messages arrive before the delay expires, they are batched together with the pending ones — reducing unnecessary LLM calls

In [3]:
# LLM-powered memory extractor: reads conversations and picks out memorable facts
memory_manager = create_memory_store_manager(
    "google_genai:gemini-2.5-flash",
    namespace=("memories",),  # Namespace acts like a folder path for organising memories
)

# Runs memory_manager in the background so the chat function stays fast.
# IMPORTANT: pass store= explicitly so the background task knows where to write memories.
# Without this, store defaults to None and no memories are ever saved.
executor = ReflectionExecutor(memory_manager, store=store)

## Step 3: Build the Chat Function

The `@entrypoint(store=store)` decorator does two things:
1. Registers this function as a LangGraph entrypoint (enabling `.invoke()` / `.ainvoke()`)
2. Injects the `store` into LangGraph's runtime context — the memory manager reads from this context when it runs in the background

Inside the function there are two paths:
- **Fast path** → LLM generates a response and returns it immediately
- **Background path** → conversation is scheduled for memory extraction after a delay

In [4]:
llm = init_chat_model("google_genai:gemini-2.5-flash")

@entrypoint(store=store)  # Wires the store into LangGraph's runtime context
def chat(message: str):
    # --- Fast path: respond to the user immediately ---
    response = llm.invoke(message)

    # --- Background path: schedule memory extraction ---
    # The conversation must be formatted as a list of OpenAI-style message dicts.
    # LangChain AIMessage objects are also accepted alongside plain dicts.
    to_process = {
        "messages": [
            {"role": "user", "content": message},
            response,  # AIMessage returned by llm.invoke()
        ]
    }

    # Submit the conversation for background processing.
    # after_seconds=0  → process as soon as possible (good for demos/testing)
    # In production, use after_seconds=1800 (30 min) so that messages from
    # the same session are batched together before the LLM reviews them.
    executor.submit(to_process, after_seconds=0.5)

    return response.content

## Step 4: Chat and Inspect Memories

Send a message that contains facts the memory manager should pick up.

In [5]:
# Send a message containing memorable facts
response = await chat.ainvoke("I like dogs. My dog's name is Fido.")
print("Bot:", response)

Bot: That's wonderful! Dogs are truly the best companions. Fido is such a classic and friendly name!

What kind of dog is Fido? I bet he brings a lot of joy to your life!


In [6]:
# WHY WE WAIT:
# executor.submit() schedules the memory manager to run in the background.
# The chat function already returned, but the LLM hasn't finished extracting
# memories yet. If we search the store immediately, it will be empty.
# We sleep briefly to let the background task complete before inspecting the store.
await asyncio.sleep(5)

# Search the store for all memories under the ("memories",) namespace.
# store.search() does a semantic (vector) search; passing no query returns everything.
memories = store.search(("memories",))

if memories:
    print(f"✓ Found {len(memories)} stored memory/memories:\n")
    for i, mem in enumerate(memories, 1):
        print(f"  [{i}] {mem.value}")
else:
    print("No memories found. Try increasing the sleep duration above.")

✓ Found 1 stored memory/memories:

  [1] {'kind': 'Memory', 'content': {'content': 'The user likes dogs.'}}


In [7]:
# Send a message containing memorable facts
response = await chat.ainvoke("I also like cats. My cat's name is Whiskers.")
print("Bot:", response)

Bot: That's an absolutely purr-fect name for a cat! Whiskers is so classic and adorable.

I'm right there with you – cats are wonderful companions!


NumPy not found in the current Python environment. The InMemoryStore will use a pure Python implementation for vector operations, which may significantly impact performance, especially for large datasets or frequent searches. For optimal speed and efficiency, consider installing NumPy: pip install numpy
